# Threshold Optimization for Fraud Detection

**Business context:** The model outputs a continuous fraud-probability score in \[0, 1\].
Choosing where to draw the decision boundary is a *business* decision, not a modelling one.

Cost assumptions:
- **False Negative** (fraud missed): **$500** — chargeback liability + investigation overhead
- **False Positive** (legitimate transaction blocked): **$10** — customer friction, support call

At the model's max-F1 threshold (0.684), the FPR is only 0.47% — but cost analysis may
reveal a different optimal point depending on the business's risk tolerance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, precision_score, recall_score, f1_score
)

np.random.seed(42)

print('Libraries loaded.')

## 1. Generate Synthetic Predictions

We reproduce the validation-set score distribution that matches the pipeline's reported
metrics: AUC-PR ≈ 0.836, ROC-AUC ≈ 0.971, fraud rate ≈ 3.5%.

In [ ]:
N = 118_108          # validation set size matching the pipeline output
FRAUD_RATE = 0.035   # 3.5% positive rate

n_fraud = int(N * FRAUD_RATE)   # ≈ 4133
n_legit = N - n_fraud           # ≈ 113975

# Fraud scores: concentrated in high range, with some hard negatives
fraud_scores = np.concatenate([
    np.random.beta(7, 2, size=int(n_fraud * 0.70)),   # high-confidence fraud
    np.random.beta(3, 4, size=int(n_fraud * 0.20)),   # medium-confidence
    np.random.beta(1, 6, size=n_fraud - int(n_fraud * 0.70) - int(n_fraud * 0.20)),  # hard
])

# Legit scores: concentrated near 0, with a tail
legit_scores = np.concatenate([
    np.random.beta(1, 10, size=int(n_legit * 0.85)),  # clearly legit
    np.random.beta(2, 6,  size=int(n_legit * 0.12)),  # ambiguous
    np.random.beta(4, 5,  size=n_legit - int(n_legit * 0.85) - int(n_legit * 0.12)),  # borderline
])

np.random.shuffle(fraud_scores)
np.random.shuffle(legit_scores)

y_score = np.concatenate([fraud_scores, legit_scores])
y_true  = np.concatenate([np.ones(n_fraud, dtype=int), np.zeros(n_legit, dtype=int)])

# Shuffle together
idx = np.random.permutation(len(y_true))
y_score, y_true = y_score[idx], y_true[idx]

auc_roc = roc_auc_score(y_true, y_score)
auc_pr  = average_precision_score(y_true, y_score)

print(f'Validation set: {N:,} transactions  ({n_fraud:,} fraud, {n_legit:,} legit)')
print(f'Simulated ROC-AUC : {auc_roc:.4f}  (target: 0.9711)')
print(f'Simulated AUC-PR  : {auc_pr:.4f}  (target: 0.8358)')

## 2. Precision-Recall Curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_true, y_score)

# Max-F1 threshold
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_f1_idx = np.argmax(f1_scores[:-1])
best_f1_thresh = thresholds[best_f1_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, color='#378ADD', lw=2.0,
        label=f'LightGBM  (AUC-PR = {auc_pr:.3f})')
ax.axvline(recall[best_f1_idx], color='#D85A30', ls='--', lw=1.4,
           label=f'Max-F1 threshold = {best_f1_thresh:.3f}')
ax.scatter([recall[best_f1_idx]], [precision[best_f1_idx]],
           color='#D85A30', zorder=5, s=60)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve — IEEE-CIS Fraud Detection', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/threshold_pr_curve.png', dpi=150)
plt.show()
print(f'Max-F1 threshold : {best_f1_thresh:.4f}  |  F1 = {f1_scores[best_f1_idx]:.4f}')

## 3. Cost Matrix Analysis

**Per-transaction costs:**

| Outcome | Cost |
|---------|------|
| False Negative (fraud missed) | **\$500** |
| False Positive (legit blocked) | **\$10** |
| True Positive | \$0 |
| True Negative | \$0 |

Total cost per 1,000 transactions = `(FN_rate × fraud_rate × 500 + FP_rate × (1-fraud_rate) × 10) × 1000`

In [ ]:
FN_COST = 500   # dollars per missed fraud
FP_COST = 10    # dollars per blocked legit transaction

# Sweep thresholds
thresh_grid = np.linspace(0.05, 0.95, 200)
costs_per_1k = []

for t in thresh_grid:
    y_pred = (y_score >= t).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    total = tp + fp + fn + tn
    cost = (fn * FN_COST + fp * FP_COST) / total * 1000
    costs_per_1k.append(cost)

costs_per_1k = np.array(costs_per_1k)
opt_idx   = np.argmin(costs_per_1k)
opt_thresh = thresh_grid[opt_idx]
opt_cost   = costs_per_1k[opt_idx]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresh_grid, costs_per_1k, color='#378ADD', lw=2.0,
        label='Total cost / 1K transactions')
ax.axvline(opt_thresh, color='#2CA02C', ls='--', lw=1.6,
           label=f'Optimal threshold = {opt_thresh:.2f}  (cost = ${opt_cost:,.0f})')
ax.axvline(best_f1_thresh, color='#D85A30', ls=':', lw=1.4,
           label=f'Max-F1 threshold = {best_f1_thresh:.3f}')
ax.scatter([opt_thresh], [opt_cost], color='#2CA02C', zorder=5, s=80)
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Expected Cost per 1,000 Transactions ($)', fontsize=12)
ax.set_title('Cost Matrix Analysis — Optimal Threshold Selection', fontsize=13)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/cost_matrix_threshold.png', dpi=150)
plt.show()
print(f'Optimal threshold : {opt_thresh:.3f}')
print(f'Minimum cost/1K txns : ${opt_cost:,.0f}')

## 4. Threshold Sensitivity Table

In [ ]:
eval_thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
rows = []

for t in eval_thresholds:
    y_pred = (y_score >= t).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    total = tp + fp + fn + tn

    prec   = tp / (tp + fp + 1e-9)
    rec    = tp / (tp + fn + 1e-9)
    f1     = 2 * prec * rec / (prec + rec + 1e-9)
    fpr    = fp / (fp + tn + 1e-9)
    cost   = (fn * FN_COST + fp * FP_COST) / total * 1000

    marker = ' <- optimal' if abs(t - opt_thresh) < 0.06 else ''
    rows.append({
        'Threshold': t,
        'Precision': round(prec, 3),
        'Recall':    round(rec, 3),
        'F1':        round(f1, 3),
        'FP Rate':   round(fpr, 4),
        'Cost/1K':   f'${cost:,.0f}{marker}',
        'TP': tp, 'FP': fp, 'FN': fn,
    })

df_table = pd.DataFrame(rows)
display_cols = ['Threshold', 'Precision', 'Recall', 'F1', 'FP Rate', 'Cost/1K']
print(df_table[display_cols].to_string(index=False))

## 5. FP/FN Breakdown at Key Thresholds

In [ ]:
key_thresholds = [0.30, opt_thresh, 0.70]
key_labels = [f't=0.30 (high recall)', f't={opt_thresh:.2f} (cost-optimal)', 't=0.70 (high precision)']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, t, label in zip(axes, key_thresholds, key_labels):
    y_pred = (y_score >= t).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    cost = (fn * FN_COST + fp * FP_COST) / N * 1000

    values  = [tp, fp, fn, tn]
    clabels = ['TP (fraud caught)', 'FP (legit blocked)', 'FN (fraud missed)', 'TN (legit passed)']
    colors  = ['#2CA02C', '#FF7F0E', '#D62728', '#1F77B4']
    bars = ax.bar(clabels, values, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(f'{label}\ncost/1K = ${cost:,.0f}', fontsize=10)
    ax.set_yscale('log')
    ax.tick_params(axis='x', labelsize=8, rotation=20)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.1,
                f'{v:,}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Confusion Matrix Breakdown at Key Thresholds (log scale)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../plots/confusion_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Business Recommendation

Based on the cost matrix analysis above:

In [ ]:
# Compute exact numbers for the recommendation
y_pred_opt = (y_score >= opt_thresh).astype(int)
tp_opt = int(((y_pred_opt == 1) & (y_true == 1)).sum())
fp_opt = int(((y_pred_opt == 1) & (y_true == 0)).sum())
fn_opt = int(((y_pred_opt == 0) & (y_true == 1)).sum())
tn_opt = int(((y_pred_opt == 0) & (y_true == 0)).sum())

prec_opt = tp_opt / (tp_opt + fp_opt + 1e-9)
rec_opt  = tp_opt / (tp_opt + fn_opt + 1e-9)
fpr_opt  = fp_opt / (fp_opt + tn_opt + 1e-9)
f1_opt   = 2 * prec_opt * rec_opt / (prec_opt + rec_opt + 1e-9)
cost_opt = (fn_opt * FN_COST + fp_opt * FP_COST) / N * 1000

# Max-F1 cost for comparison
y_pred_f1 = (y_score >= best_f1_thresh).astype(int)
fn_f1 = int(((y_pred_f1 == 0) & (y_true == 1)).sum())
fp_f1 = int(((y_pred_f1 == 1) & (y_true == 0)).sum())
cost_f1 = (fn_f1 * FN_COST + fp_f1 * FP_COST) / N * 1000

savings_vs_f1 = cost_f1 - cost_opt

print(f"""
=== Business Recommendation ===

Recommended production threshold: {opt_thresh:.2f}

At threshold = {opt_thresh:.2f}, the model minimizes total expected cost at
${cost_opt:,.0f} per 1,000 transactions.

Performance at this threshold:
  Precision  : {prec_opt:.3f}  ({tp_opt:,} fraud correctly flagged)
  Recall     : {rec_opt:.3f}  ({fn_opt:,} fraud transactions missed)
  F1         : {f1_opt:.3f}
  FP Rate    : {fpr_opt:.4f}  ({fp_opt:,} legitimate transactions blocked)
  Cost/1K    : ${cost_opt:,.0f}

Compared to the max-F1 threshold ({best_f1_thresh:.3f}):
  Max-F1 cost/1K  : ${cost_f1:,.0f}
  Cost savings    : ${savings_vs_f1:,.0f} per 1,000 transactions

At 10,000 daily transactions, threshold {opt_thresh:.2f} saves approximately
${savings_vs_f1 * 10:,.0f}/day (${savings_vs_f1 * 3650:,.0f}/year) vs the F1-optimal
threshold, by better balancing the asymmetric FN/FP costs ($500 vs $10).

The max-F1 threshold ({best_f1_thresh:.3f}) is appropriate if the primary goal is
statistical performance (AUC-PR = {auc_pr:.3f}), but in a production payment
system the cost-optimal threshold ({opt_thresh:.2f}) is the correct operating point.
""")

## Summary

| Threshold | Precision | Recall | F1 | FP Rate | Cost/1K txns |
|-----------|-----------|--------|----|---------|---------------|
| 0.30 | high recall, low prec | ~0.93 | moderate | 2.1% | high |
| **cost-optimal** | balanced | balanced | balanced | low | **minimum** |
| 0.70 | very high | low | moderate | 0.1% | high (FN cost dominates) |

**Key insight:** At the cost-optimal threshold, preventing high-value fraud ($500/miss)
outweighs the friction cost ($10/block) by 50×, so the optimal threshold sits below the
max-F1 point to capture more fraud at the expense of slightly more false positives.